# 01.07 — Data Aggregation (demand-prediction dataset)

This notebook turns the per-trip table (`data/merged/01_06_trips_weather_merged.parquet`)
into **model-ready demand datasets**: one row per *(time bucket × spatial cell)* with the
target `trip_count` plus trip, supply, temporal, weather and POI features.



We build **4 temporal × 3 spatial = 12** datasets:

| temporal            | spatial                                                           |
|---------------------|-------------------------------------------------------------------|
| `1h`, `2h`, `6h`, `24h` | H3 res 6 (`low`), H3 res 7 (`medium`), community area (`ca`)   |

**Pipeline (5 stages):**
1. **Per-trip features** — `idle_minutes`, `is_cash`, `area_type` (added once, on the raw trips).
2. **Aggregate** — group by *(time_bucket, cell)* → demand + averaged trip characteristics.
3. **Complete the grid** — add zero-demand rows for every *(bucket × active cell)*.
4. **Bucket features** — calendar (sin/cos), `season`, holidays, `area_type`, city-wide weather, and static **POI features** (what kind of place the cell is).
5. **Save** one parquet per resolution.

> **Empty buckets:** we keep every bucket for cells that have ≥ 1 trip in the year (33 H3-res6 / 160 H3-res7 / 77 community areas). In empty buckets demand and trip characteristics are **0**; weather, calendar and POI features carry their real values.

### Data Requirements:

This notebook requires the following files:
1. `01_06_trips_weather_merged.parquet` — local path: `data/merged/01_06_trips_weather_merged.parquet` — Sciebo path: `data_parquet/merged/01_06_trips_weather_merged.parquet`
2. `poi_features_community_area.parquet` — local path: `data/poi/poi_features_community_area.parquet` — produced by `01_03_Fetch_POI_Data.ipynb`
3. `poi_features_h3.parquet` — local path: `data/poi/poi_features_h3.parquet` — produced by `01_03_Fetch_POI_Data.ipynb`

Sample data is provided in the appropriate locations for testing purposes only. It will not produce meaningful results.


## Setup

In [1]:
# Imports
import os
import gc
from pathlib import Path

import numpy as np
import pandas as pd
import holidays

In [2]:
# Reset working directory to the project root (folder containing pyproject.toml)
import __main__
_nb = getattr(__main__, "__vsc_ipynb_file__", None) or os.environ.get("JPY_SESSION_NAME")
_start = Path(_nb).resolve().parent if _nb else Path.cwd()
os.chdir(next(p for p in [_start, *_start.parents] if (p / "pyproject.toml").exists()))
print(f"Working directory: {os.getcwd()}")

Working directory: /Users/hendrik/Coding/Master/AAA


In [3]:
# Load merged trip + weather data (one row per trip)
trips_all = pd.read_parquet("data/merged/01_06_trips_weather_merged.parquet")
print(f"loaded {len(trips_all):,} trips, {trips_all.shape[1]} columns")

loaded 5,369,356 trips, 39 columns


## Configuration

In [4]:
START_TIME = pd.Timestamp("2025-01-01 00:00:00")
END_TIME   = pd.Timestamp("2026-01-01 00:00:00")   # exclusive upper bound -> full year 2025

TEMPORAL_RESOLUTIONS = ["1h", "2h", "6h", "24h"]

# Idle time: a gap between two consecutive fares of the same taxi that is longer than this
# means the taxi went off-duty (overnight / days off) rather than idling for the next fare.
# Such gaps are excluded from `avg_idle_time` so they don't inflate it (see Stage 1).
SHIFT_GAP_MINUTES = 240   # 4 hours

# spatial unit name -> trip column holding the cell id
SPATIAL_UNITS = {
    "low":    "pickup_h3_res6",        # coarse H3
    "medium": "pickup_h3_res7",        # fine H3
    "ca":     "pickup_community_area",  # 77 community areas
}

# weather kept in the output (city-wide, hourly) — averaged per bucket
WEATHER_FEATURES = [
    "temperature_2m", "apparent_temperature", "precipitation",
    "rain", "snowfall", "wind_speed_10m", "cloud_cover", "is_day",
]

# Static POI features per spatial cell (built by 01_03_Fetch_POI_Data.ipynb at exactly these
# spatial units). Joined onto every bucket of a cell in Stage 4 — they describe *what kind
# of place* a cell is (airport/nightlife/medical/...), which helps explain demand patterns.
POI_CA_PATH = "data/poi/poi_features_community_area.parquet"
POI_H3_PATH = "data/poi/poi_features_h3.parquet"
POI_H3_RES_NAME = {"low": "low", "medium": "medium"}   # spatial unit -> resolution in the H3 POI file
POI_FEATURES = [
    "area_km2",
    "dist_to_nearest_airport_km", "dist_to_nearest_train_station_km", "dist_to_nearest_stadium_km",
    "train_station_per_km2", "restaurants_per_km2", "bars_and_clubs_per_km2",
    "hotels_per_km2", "hospitals_per_km2", "universities_per_km2", "attractions_per_km2",
    "poi_density_total_per_km2",
]

RUSH_HOURS = {7, 8, 9, 16, 17, 18, 19}

MONTH_TO_SEASON = {
    12: "winter",  1: "winter",  2: "winter",
     3: "spring",  4: "spring",  5: "spring",
     6: "summer",  7: "summer",  8: "summer",
     9: "autumn", 10: "autumn", 11: "autumn",
}

# area_type from the pickup community area (consistent with the CA-based pipeline)
DOWNTOWN_CA = {8, 32}    # Near North Side, The Loop
AIRPORT_CA  = {56, 76}   # Garfield Ridge (Midway), O'Hare

# columns set to 0 on empty (no-trip) rows
ZERO_FILL_COLS = [
    "trip_count", "active_taxis", "avg_idle_time",
    "avg_trip_duration", "avg_trip_distance", "avg_fare",
    "avg_trip_total", "avg_tip", "tip_rate", "share_cash_payment",
]

## Stage 1 — Per-trip features

Computed once on the raw trips (before any grouping) so aggregated statistics are correct.
- **`idle_minutes`** — *within-shift* wait before this trip: minutes since the same taxi's previous trip ended. Negative gaps (overlapping records) are clipped to 0; gaps longer than `SHIFT_GAP_MINUTES` mean the taxi went **off-duty** (overnight / days off) and are set to `NaN` so they are excluded from `avg_idle_time` instead of inflating it. Without this cut a 7-hour overnight break would count as 420 minutes of "idle", which is what produced the implausibly high `avg_idle_time` (mean ≈ 100 min, with a large pile-up at the display clip).
- **`is_cash`** — 1 if paid in cash (averages to `share_cash_payment`).
- **`area_type`** — downtown / airport / residential from the pickup community area.

In [5]:
trips_all["trip_start_timestamp"] = pd.to_datetime(trips_all["trip_start_timestamp"])
trips_all["trip_end_timestamp"]   = pd.to_datetime(trips_all["trip_end_timestamp"])

# blank taxi ids (a handful) -> NaN so they don't collapse into one fake taxi
trips_all["taxi_id"] = trips_all["taxi_id"].replace("", np.nan)

# idle time: minutes a taxi waits between consecutive fares *within a working shift*.
# A gap > SHIFT_GAP_MINUTES means the taxi went off-duty (overnight / days off) -> not idle
# waiting, so it becomes NaN (excluded from the mean) rather than a huge inflating value.
trips_all = trips_all.sort_values(["taxi_id", "trip_start_timestamp"])
prev_end    = trips_all.groupby("taxi_id")["trip_end_timestamp"].shift(1)
gap_minutes = (trips_all["trip_start_timestamp"] - prev_end).dt.total_seconds() / 60
trips_all["idle_minutes"] = gap_minutes.clip(lower=0).where(gap_minutes <= SHIFT_GAP_MINUTES)

# cash payment flag
trips_all["is_cash"] = (trips_all["payment_type"] == "Cash").astype("float64")

# area type from the pickup community area
trips_all["area_type"] = np.select(
    [trips_all["pickup_community_area"].isin(AIRPORT_CA),
     trips_all["pickup_community_area"].isin(DOWNTOWN_CA)],
    ["airport", "downtown"],
    default="residential",
)

_first_trip = gap_minutes.isna()                       # first fare of a taxi / blank id
_off_shift  = gap_minutes > SHIFT_GAP_MINUTES          # off-duty gap
print(f"idle_minutes kept (within-shift gaps): {trips_all['idle_minutes'].notna().sum():,}")
print(f"  excluded - first trip per taxi / blank id : {int(_first_trip.sum()):,}")
print(f"  excluded - off-shift gaps > {SHIFT_GAP_MINUTES} min : {int(_off_shift.sum()):,}")
print(f"within-shift idle minutes -> median {trips_all['idle_minutes'].median():.1f}, "
      f"mean {trips_all['idle_minutes'].mean():.1f}")
print("overall cash share:", round(trips_all["is_cash"].mean(), 3))
print(trips_all["area_type"].value_counts())

idle_minutes kept (within-shift gaps): 4,434,277
  excluded - first trip per taxi / blank id : 3,038
  excluded - off-shift gaps > 240 min : 932,041
within-shift idle minutes -> median 15.0, mean 42.0
overall cash share: 0.192
area_type
downtown       2270930
residential    2064866
airport        1033560
Name: count, dtype: int64


## Stage 1b — Per-resolution lookups & city-wide weather

- **active cells** — cell ids with ≥ 1 trip in the year (the spatial universe of the grid).
- **area_type per cell** — the cell's most common trip `area_type` (deterministic for community areas).
- **hourly weather** — one row per hour (weather is identical city-wide within an hour); the basis for per-bucket weather at any resolution.

In [6]:
# active cells per spatial resolution
ACTIVE_CELLS = {name: np.sort(trips_all[col].dropna().unique())
                for name, col in SPATIAL_UNITS.items()}
for name, cells in ACTIVE_CELLS.items():
    print(f"{name:7s}: {len(cells):>4} active cells")

# area_type per cell (modal trip area_type)
AREA_TYPE_LOOKUP = {
    name: trips_all.groupby(col)["area_type"].agg(lambda s: s.mode().iat[0])
    for name, col in SPATIAL_UNITS.items()
}

# one weather row per hour -> basis for per-bucket weather of any resolution
_hw = trips_all[WEATHER_FEATURES].copy()
_hw["time_bucket"] = trips_all["trip_start_timestamp"].dt.floor("h").values
HOURLY_WEATHER = (_hw.drop_duplicates("time_bucket")
                     .set_index("time_bucket")
                     .sort_index())
print("hourly weather rows:", len(HOURLY_WEATHER))
del _hw

low    :   33 active cells
medium :  160 active cells
ca     :   77 active cells
hourly weather rows: 8759


In [7]:
# keep only columns needed downstream (frees memory on this 8 GB machine)
KEEP = ["taxi_id", "trip_start_timestamp", "trip_seconds", "trip_miles",
        "fare", "tips", "trip_total", "is_cash", "idle_minutes", "area_type",
        "pickup_h3_res6", "pickup_h3_res7", "pickup_community_area"]
trips_all = trips_all[KEEP].copy()
trips_all["area_type"] = trips_all["area_type"].astype("category")
gc.collect()
print("trimmed trips_all:", trips_all.shape)

trimmed trips_all: (5369356, 13)


## Stage 1c — POI features per spatial unit

Load the static POI features (built by `01_03_Fetch_POI_Data.ipynb`) and index each one by its cell id, one table per spatial unit. They are joined onto every bucket of a cell in Stage 4. Coverage of the *active* cells should be ~100% (the H3 grid tiles the whole city); any active cell missing from the POI table is filled with no-POI defaults at join time.

In [8]:
# POI features per spatial unit, indexed by cell id (joined in Stage 4)
_poi_ca = pd.read_parquet(POI_CA_PATH)
_poi_h3 = pd.read_parquet(POI_H3_PATH)

POI_LOOKUP = {"ca": _poi_ca.set_index("community_area")[POI_FEATURES]}
for _name, _res in POI_H3_RES_NAME.items():
    POI_LOOKUP[_name] = (_poi_h3[_poi_h3["resolution"] == _res]
                         .set_index("h3_cell")[POI_FEATURES])
del _poi_ca, _poi_h3

for _name in SPATIAL_UNITS:
    _tbl = POI_LOOKUP[_name]
    _cov = np.isin(ACTIVE_CELLS[_name], _tbl.index).mean() * 100
    print(f"{_name:7s}: {len(_tbl):>5} POI cells | {_cov:5.1f}% of active cells covered")

low    :    44 POI cells | 100.0% of active cells covered
medium :   193 POI cells | 100.0% of active cells covered
ca     :    77 POI cells | 100.0% of active cells covered


## Stages 2-4 — Helper functions

In [9]:
def aggregate_trips(trips, spatial_col):
    """Stage 2: one row per (time_bucket, cell). `trips` must have a 'time_bucket' column."""
    agg = (
        trips.groupby(["time_bucket", spatial_col])
        .agg(
            trip_count         = ("fare",         "size"),     # every trip in the bucket
            active_taxis       = ("taxi_id",      "nunique"),  # ignores NaN taxi ids
            avg_idle_time      = ("idle_minutes", "mean"),
            avg_trip_duration  = ("trip_seconds", "mean"),
            avg_trip_distance  = ("trip_miles",   "mean"),
            avg_fare           = ("fare",         "mean"),
            avg_trip_total     = ("trip_total",   "mean"),
            avg_tip            = ("tips",         "mean"),
            share_cash_payment = ("is_cash",      "mean"),
        )
        .reset_index()
    )
    # tip_rate = ratio of bucket means (avg tip / avg fare)
    agg["tip_rate"] = np.where(agg["avg_fare"] > 0, agg["avg_tip"] / agg["avg_fare"], 0.0)
    return agg

In [10]:
def complete_grid(agg, buckets, spatial_col, cells):
    """Stage 3: add zero-demand rows for every (bucket x active cell)."""
    full = pd.MultiIndex.from_product(
        [buckets, cells], names=["time_bucket", spatial_col]
    ).to_frame(index=False)

    grid = full.merge(agg, on=["time_bucket", spatial_col], how="left")
    grid[ZERO_FILL_COLS] = grid[ZERO_FILL_COLS].fillna(0.0)
    grid["trip_count"]   = grid["trip_count"].astype("int64")
    grid["active_taxis"] = grid["active_taxis"].astype("int64")
    return grid

In [11]:
IL_HOLIDAYS = pd.to_datetime(list(
    holidays.country_holidays("US", subdiv="IL", years=[2025]).keys()
))


def weather_for_buckets(freq, buckets):
    """Average the hourly weather into each bucket of the given frequency."""
    wl = (HOURLY_WEATHER.resample(freq).mean()
                        .reindex(buckets)
                        .interpolate(limit_direction="both"))
    wl.index.name = "time_bucket"
    return wl.reset_index()


def _sincos(df, col, period):
    df[f"{col}_sin"] = np.sin(2 * np.pi * df[col] / period)
    df[f"{col}_cos"] = np.cos(2 * np.pi * df[col] / period)
    return df


def add_bucket_features(grid, spatial_col, area_type_lookup, weather_by_bucket, poi_table):
    """Stage 4: calendar + season + holiday + area_type + weather + POI (valid for empty cells too)."""
    tb = grid["time_bucket"]

    # generic int id for the bucket = hours since epoch (resolution-independent)
    grid["bucket_index"] = ((tb - pd.Timestamp("1970-01-01")) // pd.Timedelta(hours=1)).astype("int64")
    grid["month"]        = tb.dt.month
    grid["hour_of_day"]  = tb.dt.hour
    grid["day_of_week"]  = tb.dt.dayofweek
    grid["is_weekend"]   = (grid["day_of_week"] >= 5).astype(int)
    grid["is_rush_hour"] = grid["hour_of_day"].isin(RUSH_HOURS).astype(int)
    grid["is_holiday"]   = tb.dt.normalize().isin(IL_HOLIDAYS).astype(int)
    grid["season"]       = grid["month"].map(MONTH_TO_SEASON)

    grid = _sincos(grid, "hour_of_day", 24)
    grid = _sincos(grid, "day_of_week", 7)
    grid = _sincos(grid, "month", 12)

    # spatial feature: area type of the cell
    grid["area_type"] = grid[spatial_col].map(area_type_lookup).astype("object").fillna("residential")

    # weather is city-wide -> attach by bucket (identical for every cell in a bucket)
    grid = grid.merge(weather_by_bucket, on="time_bucket", how="left")

    # POI features are static per cell -> attach by cell (identical for every bucket of a cell)
    grid = grid.join(poi_table, on=spatial_col)
    _dist_cols = [c for c in POI_FEATURES if c.startswith("dist_")]
    _other     = [c for c in POI_FEATURES if c not in _dist_cols]
    grid[_other] = grid[_other].fillna(0.0)               # cell with no POIs -> 0 counts/area
    for _c in _dist_cols:
        grid[_c] = grid[_c].fillna(grid[_c].max())        # cell off the POI grid -> farthest
    return grid

In [12]:
# final column order (the spatial id column name is filled in per dataset)
BASE_ORDER = [
    "time_bucket", "bucket_index",
    "__SPATIAL__", "area_type",
    "trip_count",                                       # target
    "active_taxis", "avg_idle_time",                    # supply / idle proxy
    "avg_trip_duration", "avg_trip_distance", "avg_fare",
    "avg_trip_total", "avg_tip", "tip_rate", "share_cash_payment",
    "month", "hour_of_day", "day_of_week", "is_weekend",
    "is_rush_hour", "is_holiday", "season",
    "hour_of_day_sin", "hour_of_day_cos",
    "day_of_week_sin", "day_of_week_cos",
    "month_sin", "month_cos",
] + WEATHER_FEATURES + POI_FEATURES

## Stage 5 — Build & save all 12 datasets

For each *(temporal × spatial)* combination: aggregate → complete the grid → add features →
save the final dataset. Each dataset is built, written and freed in turn to stay within memory.

In [13]:
for freq in TEMPORAL_RESOLUTIONS:
    trips_all["time_bucket"] = trips_all["trip_start_timestamp"].dt.floor(freq)
    buckets = pd.date_range(START_TIME, END_TIME, freq=freq, inclusive="left")
    weather_by_bucket = weather_for_buckets(freq, buckets)

    for name, spatial_col in SPATIAL_UNITS.items():
        agg  = aggregate_trips(trips_all, spatial_col)
        grid = complete_grid(agg, buckets, spatial_col, ACTIVE_CELLS[name])
        grid = add_bucket_features(grid, spatial_col, AREA_TYPE_LOOKUP[name],
                                   weather_by_bucket, POI_LOOKUP[name])
        grid = grid[[c.replace("__SPATIAL__", spatial_col) for c in BASE_ORDER]]

        out = (f"data/aggregated/community_area/demand_ca_{freq}.parquet" if name == "ca"
               else f"data/aggregated/hexagon/demand_hex_{freq}_{name}.parquet")
        os.makedirs(os.path.dirname(out), exist_ok=True)
        grid.to_parquet(out, index=False)

        zero_pct = (grid["trip_count"] == 0).mean() * 100
        print(f"[{freq:>3} | {name:6}] rows={len(grid):>9,}  "
              f"cells={len(ACTIVE_CELLS[name]):>3}  zero-demand={zero_pct:4.1f}%  -> {out}")
        del agg, grid
        gc.collect()

[ 1h | low   ] rows=  289,080  cells= 33  zero-demand=31.5%  -> data/aggregated/hexagon/demand_hex_1h_low.parquet
[ 1h | medium] rows=1,401,600  cells=160  zero-demand=54.7%  -> data/aggregated/hexagon/demand_hex_1h_medium.parquet
[ 1h | ca    ] rows=  674,520  cells= 77  zero-demand=45.7%  -> data/aggregated/community_area/demand_ca_1h.parquet
[ 2h | low   ] rows=  144,540  cells= 33  zero-demand=23.7%  -> data/aggregated/hexagon/demand_hex_2h_low.parquet
[ 2h | medium] rows=  700,800  cells=160  zero-demand=42.8%  -> data/aggregated/hexagon/demand_hex_2h_medium.parquet
[ 2h | ca    ] rows=  337,260  cells= 77  zero-demand=32.6%  -> data/aggregated/community_area/demand_ca_2h.parquet
[ 6h | low   ] rows=   48,180  cells= 33  zero-demand=14.4%  -> data/aggregated/hexagon/demand_hex_6h_low.parquet
[ 6h | medium] rows=  233,600  cells=160  zero-demand=24.1%  -> data/aggregated/hexagon/demand_hex_6h_medium.parquet
[ 6h | ca    ] rows=  112,420  cells= 77  zero-demand=14.9%  -> data/aggreg

## Validation

Confirm the temporal resolutions now produce **different** grids (the old notebook produced an
identical hourly grid for every resolution), that there are no missing values, and preview a dataset.

In [14]:
# 1) temporal resolutions must now produce different row counts
print("Row counts per temporal resolution (H3 res6 / 'low'):")
for f in TEMPORAL_RESOLUTIONS:
    d = pd.read_parquet(f"data/aggregated/hexagon/demand_hex_{f}_low.parquet")
    exp = len(pd.date_range(START_TIME, END_TIME, freq=f, inclusive="left"))
    print(f"  {f:>3}: rows={len(d):>9,}  unique buckets={d['time_bucket'].nunique():>5}  (expected {exp})")

# 2) inspect one dataset
chk = pd.read_parquet("data/aggregated/hexagon/demand_hex_6h_low.parquet")
print("\nshape:", chk.shape)
print("total nulls:", int(chk.isna().sum().sum()))
print("zero-demand share:", round((chk["trip_count"] == 0).mean(), 3))
print("columns:", list(chk.columns))
chk.head()

Row counts per temporal resolution (H3 res6 / 'low'):
   1h: rows=  289,080  unique buckets= 8760  (expected 8760)
   2h: rows=  144,540  unique buckets= 4380  (expected 4380)
   6h: rows=   48,180  unique buckets= 1460  (expected 1460)
  24h: rows=   12,045  unique buckets=  365  (expected 365)

shape: (48180, 47)
total nulls: 0
zero-demand share: 0.144
columns: ['time_bucket', 'bucket_index', 'pickup_h3_res6', 'area_type', 'trip_count', 'active_taxis', 'avg_idle_time', 'avg_trip_duration', 'avg_trip_distance', 'avg_fare', 'avg_trip_total', 'avg_tip', 'tip_rate', 'share_cash_payment', 'month', 'hour_of_day', 'day_of_week', 'is_weekend', 'is_rush_hour', 'is_holiday', 'season', 'hour_of_day_sin', 'hour_of_day_cos', 'day_of_week_sin', 'day_of_week_cos', 'month_sin', 'month_cos', 'temperature_2m', 'apparent_temperature', 'precipitation', 'rain', 'snowfall', 'wind_speed_10m', 'cloud_cover', 'is_day', 'area_km2', 'dist_to_nearest_airport_km', 'dist_to_nearest_train_station_km', 'dist_to_nea

,time_bucket,bucket_index,pickup_h3_res6,area_type,trip_count,active_taxis,avg_idle_time,avg_trip_duration,avg_trip_distance,avg_fare,...,dist_to_nearest_train_station_km,dist_to_nearest_stadium_km,train_station_per_km2,restaurants_per_km2,bars_and_clubs_per_km2,hotels_per_km2,hospitals_per_km2,universities_per_km2,attractions_per_km2,poi_density_total_per_km2
0,2025-01-01,482136,862664197ffffff,residential,1,1,15.0,818.0,14.540,35.75,...,3.463863,5.190882,0.000000,0.412471,0.219985,0.000000,0.00000,0.000000,0.027498,0.659954
1,2025-01-01,482136,86266419fffffff,residential,0,0,0.0,0.0,0.000,0.00,...,0.770790,5.836479,0.055042,0.137605,0.055042,0.055042,0.00000,0.000000,0.027521,0.330253
2,2025-01-01,482136,8626641b7ffffff,residential,0,0,0.0,0.0,0.000,0.00,...,3.428506,4.626667,0.027470,0.247232,0.027470,0.000000,0.00000,0.000000,0.000000,0.302173
3,2025-01-01,482136,862664527ffffff,airport,5,5,0.0,1676.0,12.868,33.70,...,2.529989,4.162891,0.027462,0.631634,0.302086,0.000000,0.00000,0.000000,0.000000,0.961182
4,2025-01-01,482136,86266452fffffff,residential,0,0,0.0,0.0,0.000,0.00,...,2.695998,5.548927,0.054970,0.659643,0.137426,0.384792,0.05497,0.027485,0.027485,1.346772
